In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bib_catalog.logs;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bib_catalog.logs.process_data_pipeline_logs
(
    notebook_name STRING,
    pipeline_name STRING,
    source_name STRING,
    layer STRING,
    status STRING,
    records_processed BIGINT,
    message STRING,
    environment STRING,
    log_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%run ../configs/common_configs

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp

In [0]:
def log_pipeline_event(
    notebook_name,
    pipeline_name,
    source_name,
    layer,
    status,
    records_processed,
    message,
    environment
):
    
    log_data = [

        Row(
            notebook_name=notebook_name,
            pipeline_name=pipeline_name,
            source_name=source_name,
            layer=layer,
            status=status,
            records_processed=records_processed,
            message=message,
            environment=environment
        )
    ]

    log_df = spark.createDataFrame(log_data)

    log_df = log_df.withColumn(
        "log_timestamp",
        current_timestamp()
    )

    # -------------------------
    # Unity Catalog Logging
    # -------------------------

    (
        log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(
            "bib_catalog.logs.process_data_pipeline_logs"
        )
    )

    # -------------------------
    # ADLS Logging
    # -------------------------

    (
        log_df.write
        .format("delta")
        .mode("append")
        .save(
            f"abfss://{environment}@{storage_account}.dfs.core.windows.net/"
            f"{layer}/{domain}/{source_name}-logs"
        )
    )

    print(
        f"{status} log written for "
        f"{source_name}"
    )